# 讓 Agent 接得住前後文這份教材的主題是同一段對話如何延續。基礎模組不另外拆解，重點放在 `session_id` 與 `memory.turns`。

## ? Colab ????????? Colab ???????????????????????????????????????

In [ ]:
!git clone https://github.com/R300-AI/Agentic-SDK.git%cd Agentic-SDK

## 載入流程元件這次只需要一條簡單流程，方便觀察對話記憶。

In [ ]:
from agentic_sdk import Workflowfrom agentic_sdk.modules import DirectAnswerAction, KeywordRetrieve, PassThroughPerceive

## 先決定這段對話的識別碼同一個使用者、同一段對話，應該使用同一個 `session_id`。換掉它，就等於開一段新對話。

In [ ]:
session_id = 'demo-user-001'session_id

## 建立可以重複使用的流程流程本身不用每輪重建。這裡把查詢資料直接放進流程裡，因為本章真正要看的不是查詢設定，而是同一個 `session_id` 如何累積記憶。

In [ ]:
workflow = Workflow(    workflow_name='多輪問答 Agent',    perceive=PassThroughPerceive(),    retrieve=KeywordRetrieve(items=[        {'keywords': ['專案代號', 'aurora'], 'content': '使用者提到的專案代號是 Aurora。'},        {'keywords': ['會議', '明天'], 'content': '明天會議需要準備專案摘要。'},    ]),    action=DirectAnswerAction(),)

## 第一輪：先告訴 Agent 一件事第一輪會把使用者訊息和 Agent 回覆保存到這個 `session_id` 底下。

In [ ]:
first = workflow.run('請記住，這次專案代號是 Aurora。', session_id=session_id)print(first.final_message)

## 第二輪：接著問後續問題第二輪仍然使用同一個 `session_id`，所以記憶裡會保留前一輪內容。

In [ ]:
second = workflow.run('那明天會議我要準備什麼？', session_id=session_id)print(second.final_message)

## 查看保存下來的對話紀錄這裡看的是記憶類型保存的完整 turn 順序，不是單次執行的中間資料。

In [ ]:
memory = second.memoryfor index, turn in enumerate(memory.turns, start=1):    print(index, turn.role, '=>', turn.content)

## 換一個對話識別碼看看換成新的 `session_id`，就會得到一段新的對話記憶。這可以避免不同使用者或不同任務互相污染。

In [ ]:
new_session = workflow.run('剛剛的專案代號是什麼？', session_id='another-session')print(new_session.final_message)print('turns:', len(new_session.memory.turns))